# 02 — Example Run: 1 km, 10 cm ID, Q-in(t) & fixed p_out
This notebook imports the solver and runs a short transient case.

If you're running this in a fresh environment, ensure `hb_transient.py` is on the path.
Alternatively, copy the solver cell from the previous notebook here.

In [ ]:
from hb_transient import HBParams, PipeParams, FluidCompress, Numerics, build_pipe, HBTransient1D
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# --- Parameters ---
L = 1000.0         # m
D = 0.10           # m
dx = 10.0          # m (box length)
dt = 0.02          # s
t_end = 2.0        # s  (shorter by default for quick test; raise as needed)

pipe = build_pipe(L, D)
fluid = FluidCompress(rho0=1200.0, p0=1.0e5, B=1.5e9)
hb = HBParams(tau_y=5.0, K=0.5, n=0.6)
num = Numerics(dx=dx, dt=dt, cfl=0.5)
p_out = 1.0e5  # Pa

def Q_in(t):
    # ramp + small oscillation
    Q = 0.012 * min(t/2.0, 1.0) + 0.001 * np.sin(2*np.pi*0.5*t)
    return Q

solver = HBTransient1D(pipe, fluid, hb, num, p_out)

times = [0.0]
p_hist = [solver.p.copy()]
t = 0.0
while t < t_end - 1e-12:
    solver.step(t, Q_in)
    t += dt
    times.append(t)
    p_hist.append(solver.p.copy())

import pandas as pd
x = solver.xc
df = pd.DataFrame({'x_m': x, 'p_last_Pa': solver.p})
df.head()

In [ ]:
# Quick plot of the final pressure
plt.figure()
plt.plot(solver.xc, solver.p/1e5, label=f't={times[-1]:.2f} s')
plt.xlabel('x [m]')
plt.ylabel('Pressure [bar]')
plt.title('HB transient: final pressure')
plt.legend()
plt.tight_layout()
plt.show()